In [1]:
import os, sys, importlib
import pandas as pd
import json
import networkx as nt
import matplotlib.pyplot as plt

## data

In [2]:
path_table="../results/table_clean.csv"
df_all=pd.read_csv(path_table)
display(df_all)

,title_theme,title_index,author,theme,cycle,collection,mss,ff,page
0,Heroica,Heroica,Flavius Philostratus,Classical Romances,Cycle of Troy,Royal,16. C. xxiii.,ff. 2-69 b,1
1,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Royal,16. C. iv. A. B.,NaN,2
2,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Royal,16. D. iii. A. B.,NaN,2
3,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Harley,Harley 5662.,ff. 1-56,2
4,Dictys Cretensis,Dictys Cretensis,NaN,Classical Romances,Cycle of Troy,Harley,Harley 3514,NaN,9
...,...,...,...,...,...,...,...,...,...
578,NaN,A Propecye,NaN,NaN,NaN,Additional,Add. 27879 f 239 b,NaN,327
579,NaN,Robin Hood's Garland,NaN,NaN,NaN,Additional,Add. 28638,NaN,520
580,NaN,Ballads of Scotland,NaN,NaN,NaN,Additional,"Add. 29408, 29409",NaN,537
581,NaN,Two Charlemagne Romances,NaN,NaN,NaN,Additional,Add. 31042,"ff66 b-79 b, 82-94",953


## vis1.1 : collection network (edge==sum coocurrences of titles)

In [30]:
df=df_all.copy()
df=df[df["collection"].notna()]
display(df.head())
print(len(df_all),'=>', len(df))

print(df[['title_index','collection']].value_counts(dropna=False),"\n")
print(df[['title_index','mss']].value_counts(dropna=False))


,title_theme,title_index,author,theme,cycle,collection,mss,ff,page
0,Heroica,Heroica,Flavius Philostratus,Classical Romances,Cycle of Troy,Royal,16. C. xxiii.,ff. 2-69 b,1
1,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Royal,16. C. iv. A. B.,NaN,2
2,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Royal,16. D. iii. A. B.,NaN,2
3,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Harley,Harley 5662.,ff. 1-56,2
4,Dictys Cretensis,Dictys Cretensis,NaN,Classical Romances,Cycle of Troy,Harley,Harley 3514,NaN,9


583 => 576
title_index                collection
Alexandreis                Additional    12
Turpin's Chronicle         Harley        12
                           Cotton        10
Alexander the Great        Royal          9
Historia Regum Britanniae  Cotton         9
                                         ..
Brut y Brenhinoedd         Cotton         1
Brutus                     Cotton         1
Cassandra                  Sloane         1
Chanson d'Aspremont        Lansdowne      1
Trójumanna saga            Additional     1
Name: count, Length: 327, dtype: int64 

title_index          mss              
Havelok              13. A. xxi.          4
Alexandreis          8. B. iv.            4
Turpin's Chronicle   Nero A. xi.          4
Alexandreis          Harley 4745          4
                     Burney 312           4
                                         ..
Aspremont            15. E. vi.           1
                     Add. 10,508.         1
Ballads of Scotland  Add. 29408, 29

In [31]:
'''
思路：
每个 collection 是节点
如果两个 collection 都包含同一个 title → edge
edge weight = 共享 title 在两边的出现次数之和（或者乘积）
edge label = title
'''

from itertools import combinations
from collections import defaultdict
import pandas as pd

# 按 collection × title_index 统计
coll_title_counts = df.groupby(['collection','title_index']).size().reset_index(name='count')
print(coll_title_counts)

# 初始化字典，key=(coll1, coll2)
edge_dict = defaultdict(lambda: {'weight':0, 'titles':[]})

# 按 title_index 分组
title_groups = coll_title_counts.groupby('title_index')

for title_index, group in title_groups:
    collections = list(group['collection'])
    counts = list(group['count'])
    
    if len(collections) > 1:
        for (i,j) in combinations(range(len(collections)),2):
            # key 按字母顺序保证唯一性
            key = tuple(sorted([collections[i], collections[j]]))
            edge_dict[key]['weight'] += counts[i] + counts[j]   # 累加权重
            edge_dict[key]['titles'].append(title_index)        # 累加共享 title

# 转成 DataFrame
edges = []
for (coll1, coll2), data in edge_dict.items():
    edges.append({
        'source': coll1,
        'target': coll2,
        'weight': data['weight'],
        'titles': data['titles']
    })

edge_df = pd.DataFrame(edges)
# display(edge_df)

# 可选：只保留 weight >= 2 的边
edge_df_filtered = edge_df[edge_df['weight'] >= 2]
display(edge_df_filtered)


     collection                 title_index  count
0    Additional                  A Propecye      1
1    Additional              Alexander saga      2
2    Additional                 Alexandreis     12
3    Additional            Amis and Amylion      1
4    Additional          Apollonius of Tyre      2
..          ...                         ...    ...
322      Sloane           Quatre fils Aimon      1
323      Sloane          Robyn and Gandelyn      1
324      Sloane           Sydrac and Boctus      1
325      Sloane             Tale of Gamelyn      2
326      Sloane  Whole Prophecy of Scotland      1

[327 rows x 3 columns]


,source,target,weight,titles
0,Arundel,Burney,2,[Alexander the Great]
1,Arundel,Cotton,48,"[Alexander the Great, Apollonius of Tyre, Hist..."
2,Arundel,Harley,46,"[Alexander the Great, Historia Regum Britannia..."
3,Arundel,Royal,45,"[Alexander the Great, Apollonius of Tyre, Hist..."
4,Arundel,Sloane,16,"[Alexander the Great, Apollonius of Tyre, Hist..."
5,Burney,Cotton,11,"[Alexander the Great, Dares Phrygius]"
6,Burney,Harley,22,"[Alexander the Great, Alexandreis, Dares Phryg..."
7,Burney,Royal,29,"[Alexander the Great, Alexandreis, Dares Phryg..."
8,Burney,Sloane,6,"[Alexander the Great, Dares Phrygius]"
9,Cotton,Harley,79,"[Alexander the Great, Dares Phrygius, Fall of ..."


In [33]:
import numpy as np
# --------------NODE SIZE------------------
# 每个 collection 的文本数量
node_sizes_raw = df.groupby('collection')['title_index'].nunique()

# normalize 到 20~60 大小
min_size, max_size = 10, 30
node_sizes = ((node_sizes_raw - node_sizes_raw.min()) / (node_sizes_raw.max() - node_sizes_raw.min()) * (max_size - min_size) + min_size).to_dict()
print(node_sizes)

#-------------NODE COLOR--------------------
from matplotlib import cm, colors

collections = list(df['collection'].unique())

# viridis 颜色映射
cmap = cm.get_cmap('tab20', len(collections))
color_map = {coll: colors.to_hex(cmap(i)) for i, coll in enumerate(collections)}

# ------------network--------------------
from pyvis.network import Network

# notebook=True 在 jupyter 直接显示
net = Network(height='800px', width='100%', notebook=True)

for coll in collections:
    net.add_node(
        coll,
        label=coll,
        title=f"{coll}: {node_sizes_raw[coll]} texts",
        color=color_map[coll],
        size=node_sizes[coll],
        font={
            "size": 25,        # 字体大小
            "face": "arial",   # 字体类型
            "color": "black", # 颜色
        }
    )
    
max_titles_to_show=2

for _, row in edge_df_filtered.iterrows():    
    if len(row['titles'])<max_titles_to_show:
        hover_text= f"{', '.join(row['titles'])}({row['weight']})" 
    else :
        hover_text= f"{', '.join(row['titles'][:max_titles_to_show])}...({row['weight']})" 

    
    net.add_edge(
        row['source'],
        row['target'],
        value=row['weight']/5,
        title=hover_text,
        font={"size": 5, "align":"middle"},
        color="lightgray",
        
        physics=True
    )



# -------------LAYOUT--------------------
net.force_atlas_2based(
    gravity=-200,          # 整体节点拉开的力度，负值越大，节点分得越开
    central_gravity=0.05,  # 越小，节点越靠中心；越大，节点聚到中心
    spring_length=250,     # 弹簧长度，越大，节点间隔越远
    spring_strength=0.05,  # 弹簧刚度，控制边对节点靠近的作用力
    damping=0.4            # 收敛阻尼
)

#-------------------SHOW---------------------
# 生成 HTML
path_html='../figs/collection_network.html'
net.show(path_html)

## ----------------add title---------------

# 自动在 body 开始处插入标题
title_text = "Inter-Collection Network of Shared Manuscript Texts"
with open(path_html, 'r', encoding='utf-8') as f:
    html_content = f.read()

# 在 <body> 标签后插入标题
html_content = html_content.replace(
    '<body>',
    f'<body>\n<h2 style="text-align:center">{title_text}</h2>'
)
# 保存回 HTML 文件
with open(path_html, 'w', encoding='utf-8') as f:
    f.write(html_content)

{'Additional': 30.0, 'Arundel': 11.801801801801801, 'Burney': 10.18018018018018, 'Cotton': 18.28828828828829, 'Egerton': 10.0, 'Harley': 19.90990990990991, 'Lansdowne': 11.621621621621621, 'Royal': 18.82882882882883, 'Sloane': 11.801801801801801}
../figs/collection_network.html


C:\Users\yeliu\AppData\Local\Temp\ipykernel_12060\1642905073.py:17: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('tab20', len(collections))


## vis1.2 : mss network: connecté si partage le meme texte
QR : comment les collections se construisent? 

si le regroupement dépend de la similarité entre les mss (==représenté par le partage des mêmes textes) ?

on 1) associe les mss qui enregistre le meme texte 
2) colore les mss selon la collection


si oui, un cluster a une couleur dominante
si non, le cluster a plusieurs couleurs

==> la collection ne se fait pas selon le contenu.


In [34]:
from itertools import combinations
from collections import defaultdict
import pandas as pd

# 按 collection × title_index 统计
coll_title_counts = df.groupby(["collection",'mss','title_index']).size().reset_index(name='count')
print(coll_title_counts)

# 初始化字典，key=(coll1, coll2)
edge_dict = defaultdict(lambda: {'weight':0, 'titles':[]})

# 按 title_index 分组
title_groups = coll_title_counts.groupby('title_index')

for title_index, group in title_groups:
    collections = list(group['mss'])
    counts = list(group['count'])
    
    if len(collections) > 1:
        for (i,j) in combinations(range(len(collections)),2):
            # key 按字母顺序保证唯一性
            key = tuple(sorted([collections[i], collections[j]]))
            edge_dict[key]['weight'] += counts[i] + counts[j]   # 累加权重
            edge_dict[key]['titles'].append(title_index)        # 累加共享 title

# 转成 DataFrame
edges = []
for (coll1, coll2), data in edge_dict.items():
    edges.append({
        'source': coll1,
        'target': coll2,
        'weight': data['weight'],
        'titles': data['titles']
    })

edge_df = pd.DataFrame(edges)
# display(edge_df)  

# 可选：只保留 weight >= 2 的边
edge_df_filtered = edge_df[edge_df['weight'] >= 2]
display(edge_df_filtered)


     collection           mss                                title_index  \
0    Additional  Add. 10,038.                        Titus and Vespasian   
1    Additional  Add. 10,094.                             Dares Phrygius   
2    Additional  Add. 10,286.                          Sydrac and Boctus   
3    Additional  Add. 10,289.                                     Juglet   
4    Additional  Add. 10,289.                        Titus and Vespasian   
..          ...           ...                                        ...   
497      Sloane   Sloane 289.                  Historia Regum Britanniae   
498      Sloane  Sloane 3991.  Letter of Dindimus to Alexander the Great   
499      Sloane    Sloane 457                                  Cassandra   
500      Sloane   Sloane 780.                         Life of Robin Hood   
501      Sloane    Sloane 960                          Quatre fils Aimon   

     count  
0        3  
1        1  
2        1  
3        1  
4        3  
..     ..

,source,target,weight,titles
0,"Add. 11,238.","Add. 24,969.",2,[Alexander saga]
1,Arundel 123.,Burney 280.,2,[Alexander the Great]
2,Arundel 123.,Cleopatra D. v.,2,[Alexander the Great]
3,Arundel 123.,Galba E. xi.,2,[Alexander the Great]
4,Arundel 123.,Nero D. viii.,2,[Alexander the Great]
...,...,...,...,...
1642,Add. 30864,Arundel 230,4,[Voeux du Paon]
1643,Add. 30864,Harley 3992,4,[Voeux du Paon]
1644,Arundel 230,Harley 3992,4,[Voeux du Paon]
1645,Sloane 1802,Vespasian E. viii.,2,[Whole Prophecy of Scotland]


In [26]:
coll_title_counts[coll_title_counts['title_index']=='Titus and Vespasian']

,collection,mss,title_index,count
0,Additional,"Add. 10,038.",Titus and Vespasian,3
4,Additional,"Add. 10,289.",Titus and Vespasian,3
118,Additional,Add. 31042,Titus and Vespasian,3
184,Cotton,Caligula A. ii.,Titus and Vespasian,3
259,Cotton,Vespasian E. xvi.,Titus and Vespasian,3
440,Royal,16. E. viii.,Titus and Vespasian,3


In [62]:
import numpy as np
# --------------NODE SIZE------------------
# 每个 collection 的文本数量
node_sizes_raw = df.groupby('mss')['title_index'].nunique()

# normalize 到 20~60 大小
min_size, max_size = 20, 60
node_sizes = ((node_sizes_raw - node_sizes_raw.min()) / (node_sizes_raw.max() - node_sizes_raw.min()) * (max_size - min_size) + min_size).to_dict()
print(node_sizes)

#-------------NODE COLOR--------------------
from matplotlib import cm, colors
mss_collection = df.drop_duplicates('mss').set_index('mss')['collection'].to_dict()
collections = df['collection'].unique()

cmap = cm.get_cmap('tab20', len(collections))
collection_colors = {
    coll: colors.to_hex(cmap(i))
    for i, coll in enumerate(collections)
}
# {'Royal': '#1f77b4',
#  'Harley': '#ff7f0e',
#  'Burney': '#98df8a',
#  'Additional': '#ff9896',
#  'Cotton': '#8c564b',
#  'Sloane': '#e377c2',
#  'Lansdowne': '#c7c7c7',
#  'Arundel': '#dbdb8d',
#  'Egerton': '#9edae5'}


# ------------network--------------------
from pyvis.network import Network
net = Network(height='800px', width='100%', notebook=True)


for mss in node_sizes_raw.index:
    coll = mss_collection[mss]
    net.add_node(
        mss,
        label=mss,
        title=f"{mss}({coll}): {node_sizes_raw[mss]} texts",
        color=collection_colors[coll],
        size=node_sizes[mss]
    )
    
    
max_titles_to_show=2

for _, row in edge_df_filtered.iterrows():    
    if len(row['titles'])<max_titles_to_show:
        hover_text= f"{', '.join(row['titles'])}({row['weight']})" 
    else :
        hover_text= f"{', '.join(row['titles'][:max_titles_to_show])}...({row['weight']})" 

    
    net.add_edge(
        row['source'],
        row['target'],
        value=row['weight']/5,
        title=hover_text,
        font={"size": 5, "align":"middle"},
        color="lightgray",
        physics=True
    )

# -------------LAYOUT--------------------
net.force_atlas_2based(
    gravity=-200,          # 整体节点拉开的力度，负值越大，节点分得越开
    central_gravity=0.05,  # 越小，节点越靠中心；越大，节点聚到中心
    spring_length=250,     # 弹簧长度，越大，节点间隔越远
    spring_strength=0.05,  # 弹簧刚度，控制边对节点靠近的作用力
    damping=0.4            # 收敛阻尼
)

#-------------------SHOW---------------------
# 生成 HTML
path_html='../figs/mss_network_by_collection.html'
net.show(path_html)

## ----------------add title---------------

# 自动在 body 开始处插入标题
title_text = "Inter-Manuscript Network of Shared Texts colored by collection"
with open(path_html, 'r', encoding='utf-8') as f:
    html_content = f.read()

# 在 <body> 标签后插入标题
html_content = html_content.replace(
    '<body>',
    f'<body>\n<h2 style="text-align:center">{title_text}</h2>'
)
# 保存回 HTML 文件
with open(path_html, 'w', encoding='utf-8') as f:
    f.write(html_content)

{'10. A. x.': 20.0, '12. C. iv.': 20.0, '12. C. xii.': 26.666666666666664, '12. C. xx.': 20.0, '12. D. iii.': 23.333333333333332, '13. A. i.': 20.0, '13. A. iii.': 23.333333333333332, '13. A. iv.': 20.0, '13. A. v.': 26.666666666666664, '13. A. xiv.': 20.0, '13. A. xviii.': 20.0, '13. A. xxi.': 23.333333333333332, '13. C. xii.': 23.333333333333332, '13. D. i.': 23.333333333333332, '13. D. ii.': 20.0, '13. D. v.': 20.0, '13. E. i.': 20.0, '13. E. ix.': 20.0, '14. C. i.': 20.0, '14. C. xi.': 20.0, '14. E. ii.': 23.333333333333332, '14. E. iii.': 23.333333333333332, '15. A. x.': 20.0, '15. A. xxii.': 23.333333333333332, '15. B. xi.': 23.333333333333332, '15. C. vi.': 20.0, '15. C. xvi.': 26.666666666666664, '15. E. v.': 20.0, '15. E. vi.': 46.666666666666664, '16. C. iv. A. B.': 20.0, '16. C. xxiii.': 20.0, '16. D. iii. A. B.': 20.0, '16. E.': 20.0, '16. E. viii.': 20.0, '16. F. ix.': 20.0, '16. F. v.': 20.0, '16. G. ii.': 20.0, '17. B. xliii.': 20.0, '17. D. xv.': 20.0, '17. E. ii.': 20.

C:\Users\yeliu\AppData\Local\Temp\ipykernel_12060\985019461.py:16: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('tab20', len(collections))


../figs/mss_network_by_collection.html


## vis1.3 xxx mss belongs to several themes

In [28]:
from itertools import combinations
from collections import defaultdict
import pandas as pd

# 按 collection × title_index 统计
coll_title_counts = df.groupby(["theme",'mss','title_index']).size().reset_index(name='count')
display(coll_title_counts.head())

# 初始化字典，key=(coll1, coll2)
edge_dict = defaultdict(lambda: {'weight':0, 'titles':[]})

# 按 title_index 分组
title_groups = coll_title_counts.groupby('title_index')

for title_index, group in title_groups:
    collections = list(group['mss'])
    counts = list(group['count'])
    
    if len(collections) > 1:
        for (i,j) in combinations(range(len(collections)),2):
            # key 按字母顺序保证唯一性
            key = tuple(sorted([collections[i], collections[j]]))
            edge_dict[key]['weight'] += counts[i] + counts[j]   # 累加权重
            edge_dict[key]['titles'].append(title_index)        # 累加共享 title

# 转成 DataFrame
edges = []
for (coll1, coll2), data in edge_dict.items():
    edges.append({
        'source': coll1,
        'target': coll2,
        'weight': data['weight'],
        'titles': data['titles']
    })

edge_df = pd.DataFrame(edges)
# display(edge_df)

# 可选：只保留 weight >= 2 的边
edge_df_filtered = edge_df[edge_df['weight'] >= 2]
display(edge_df_filtered)


,theme,mss,title_index,count
0,Allegorical and Didactic Romance,14. E. ii.,Le Chemin de Vaillance,1
1,Allegorical and Didactic Romance,14. E. ii.,Livre de l'Ordre de Chevalerie,1
2,Allegorical and Didactic Romance,16. F. v.,Sydrac and Boctus,1
3,Allegorical and Didactic Romance,19. A. xviii.,Roman de la Rose,1
4,Allegorical and Didactic Romance,19. B. xii.,Roman de la Rose,1


,source,target,weight,titles
0,"Add. 11,238.","Add. 24,969.",2,[Alexander saga]
1,15. A. x.,8. B. iv.,8,[Alexandreis]
2,15. A. x.,"Add. 17,084",8,[Alexandreis]
3,15. A. x.,"Add. 18,217",8,[Alexandreis]
4,15. A. x.,"Add. 22,821",8,[Alexandreis]
...,...,...,...,...
928,Add. 30864,Harley 3992,8,"[Voeux du Paon, Voeux du Paon, Voeux du Paon, ..."
929,Add. 30864,Add. 30864,2,[Voeux du Paon]
930,Arundel 230,Harley 3992,8,"[Voeux du Paon, Voeux du Paon, Voeux du Paon, ..."
931,Arundel 230,Arundel 230,2,[Voeux du Paon]


In [30]:
mss_theme

{'16. C. xxiii.': 'Classical Romances',
 '16. C. iv. A. B.': 'Classical Romances',
 '16. D. iii. A. B.': 'Classical Romances',
 'Harley 5662.': 'Classical Romances',
 'Harley 3514': 'Classical Romances',
 'Burney 170': 'Classical Romances',
 'Add. 15,429': 'Classical Romances',
 '6. C. viii.': 'Classical Romances',
 '10. A. x.': 'Classical Romances',
 '13. A. v.': 'Classical Romances',
 '15. A. xxii.': 'Classical Romances',
 '15. B. xi.': 'Classical Romances',
 'Caligula B. vii.': 'Classical Romances',
 'Vitellius A. xiii.': 'Classical Romances',
 'Vitellius C. viii.': 'Classical Romances',
 'Vespasian B. xxv.': 'Classical Romances',
 'Cleopatra B. v.': 'Classical Romances',
 'Harley 641.': 'Classical Romances',
 'Burney 216.': 'Classical Romances',
 'Burney 280.': 'Classical Romances',
 'Sloane 1619': 'Classical Romances',
 'Add. 10,094.': 'Classical Romances',
 'Add. 15,042': 'Classical Romances',
 'Add. 19,709.': 'Classical Romances',
 'Harley 4482': 'Classical Romances',
 'Add. 308

In [29]:
import numpy as np
# --------------NODE SIZE------------------
# 每个 collection 的文本数量
node_sizes_raw = df.groupby('mss')['title_index'].nunique()

# normalize 到 20~60 大小
min_size, max_size = 20, 60
node_sizes = ((node_sizes_raw - node_sizes_raw.min()) / (node_sizes_raw.max() - node_sizes_raw.min()) * (max_size - min_size) + min_size).to_dict()
print(node_sizes)

#-------------NODE COLOR--------------------
from matplotlib import cm, colors
mss_theme = df.drop_duplicates('mss').set_index('mss')['theme'].to_dict()
themes = df['theme'].unique()

cmap = cm.get_cmap('tab20', len(themes))
theme_colors = {
    coll: colors.to_hex(cmap(i))
    for i, coll in enumerate(themes)
}
print(theme_colors)

# ------------network--------------------
from pyvis.network import Network
net = Network(height='800px', width='100%', notebook=True)


for mss in node_sizes_raw.index:
    theme = mss_theme[mss]
    net.add_node(
        mss,
        label=mss,
        title=f"{mss}({theme}) : {node_sizes_raw[mss]} texts",
        color=theme_colors[theme],
        size=node_sizes[mss]
    )
    
    
max_titles_to_show=2

for _, row in edge_df_filtered.iterrows():    
    if len(row['titles'])<max_titles_to_show:
        hover_text= f"{', '.join(row['titles'])}({row['weight']})" 
    else :
        hover_text= f"{', '.join(row['titles'][:max_titles_to_show])}...({row['weight']})" 

    
    net.add_edge(
        row['source'],
        row['target'],
        value=row['weight']/5,
        title=hover_text,
        font={"size": 5, "align":"middle"},
        color="lightgray",
        physics=True
    )

# -------------LAYOUT--------------------
net.force_atlas_2based(
    gravity=-200,          # 整体节点拉开的力度，负值越大，节点分得越开
    central_gravity=0.05,  # 越小，节点越靠中心；越大，节点聚到中心
    spring_length=250,     # 弹簧长度，越大，节点间隔越远
    spring_strength=0.05,  # 弹簧刚度，控制边对节点靠近的作用力
    damping=0.4            # 收敛阻尼
)

#-------------------SHOW---------------------
# 生成 HTML
path_html='../figs/mss_network_theme.html'
net.show(path_html)

## ----------------add title---------------

# 自动在 body 开始处插入标题
title_text = "Inter-Manuscript Network of Shared Texts colored by theme"
with open(path_html, 'r', encoding='utf-8') as f:
    html_content = f.read()

# 在 <body> 标签后插入标题
html_content = html_content.replace(
    '<body>',
    f'<body>\n<h2 style="text-align:center">{title_text}</h2>'
)
# 保存回 HTML 文件
with open(path_html, 'w', encoding='utf-8') as f:
    f.write(html_content)

{'10. A. x.': 20.0, '12. C. xii.': 20.0, '12. C. xx.': 20.0, '12. D. iii.': 20.0, '13. A. v.': 20.0, '13. A. xiv.': 20.0, '13. A. xviii.': 20.0, '13. A. xxi.': 28.0, '13. C. xii.': 20.0, '13. D. i.': 20.0, '13. E. i.': 20.0, '14. C. i.': 20.0, '14. C. xi.': 20.0, '14. E. ii.': 28.0, '14. E. iii.': 28.0, '15. A. x.': 20.0, '15. A. xxii.': 28.0, '15. B. xi.': 28.0, '15. C. xvi.': 28.0, '15. E. v.': 20.0, '15. E. vi.': 60.0, '16. C. iv. A. B.': 20.0, '16. C. xxiii.': 20.0, '16. D. iii. A. B.': 20.0, '16. E. viii.': 20.0, '16. F. ix.': 20.0, '16. F. v.': 20.0, '16. G. ii.': 20.0, '17. B. xliii.': 20.0, '17. D. xv.': 20.0, '18. B. ii.': 20.0, '18. C. ii.': 20.0, '18. D. ii.': 28.0, '18. D. vi.': 20.0, '19. A. xviii.': 20.0, '19. B. vii.': 20.0, '19. B. xii.': 20.0, '19. B. xiii.': 20.0, '19. C. vii.': 20.0, '19. C. viii.': 20.0, '19. D. i.': 20.0, '19. E. ii.': 20.0, '19. E. iii.': 20.0, '20. A. xvii.': 20.0, '20. B. xix.': 28.0, '20. C. ii.': 28.0, '20. C. vi.': 20.0, '20. D. ii.': 20.0, '

C:\Users\yeliu\AppData\Local\Temp\ipykernel_21292\1581401131.py:16: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('tab20', len(themes))


## vis2.1 XXX: connecter les textes dans la meme collection, coloré selon le thème


RQ : si collection se fait par le theme?

TOOOOOO dense!!!!




In [3]:
df=df_all.copy()
df_clean=df[df["collection"].notna()]
print(len(df_all), len(df_clean))
df_clean=df_clean.dropna(subset='theme')
print(df_clean.theme.value_counts(dropna=False),"\n")
df_clean

583 576
theme
Classical Romances                  146
British and English Traditions      109
French Traditions                    47
Miscellaneous Romance                38
Appendix                             38
Allegorical and Didactic Romance     25
Name: count, dtype: int64 



,title_theme,title_index,author,theme,cycle,collection,mss,ff,page
0,Heroica,Heroica,Flavius Philostratus,Classical Romances,Cycle of Troy,Royal,16. C. xxiii.,ff. 2-69 b,1
1,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Royal,16. C. iv. A. B.,NaN,2
2,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Royal,16. D. iii. A. B.,NaN,2
3,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Harley,Harley 5662.,ff. 1-56,2
4,Dictys Cretensis,Dictys Cretensis,NaN,Classical Romances,Cycle of Troy,Harley,Harley 3514,NaN,9
...,...,...,...,...,...,...,...,...,...
404,Turpin's Chronicle,Turpin's Chronicle,NaN,Appendix,NaN,Additional,Add. 6924.,ff. 297-303,950
405,Turpin's Chronicle,Turpin's Chronicle,NaN,Appendix,NaN,Additional,"Add. 12,213.",ff. 1-184 b,950
406,Turpin's Chronicle,Turpin's Chronicle,NaN,Appendix,NaN,Additional,"Add. 17,920.",ff. 6 b-19 b,950
407,Turpin's Chronicle,Turpin's Chronicle,NaN,Appendix,NaN,Additional,"Add. 19,513.",ff. 141-163,950


### par collection

In [5]:
df=df_clean[df_clean['collection']=="Royal"]
display(df.head())

,title_theme,title_index,author,theme,cycle,collection,mss,ff,page
0,Heroica,Heroica,Flavius Philostratus,Classical Romances,Cycle of Troy,Royal,16. C. xxiii.,ff. 2-69 b,1
1,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Royal,16. C. iv. A. B.,NaN,2
2,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Royal,16. D. iii. A. B.,NaN,2
7,Dares Phrygius,Dares Phrygius,NaN,Classical Romances,Cycle of Troy,Royal,6. C. viii.,ff. 123-133 b,12
8,Dares Phrygius,Dares Phrygius,NaN,Classical Romances,Cycle of Troy,Royal,10. A. x.,ff. 188-192 b,12


In [6]:
text_mss = df.groupby('title_index')['mss'].apply(lambda x: set(x)).to_dict()
print(text_mss)

from itertools import combinations
edges = []
for text1, text2 in combinations(text_mss.keys(), 2):

    shared_mss = text_mss[text1].intersection(text_mss[text2])

    weight = len(shared_mss)

    if weight > 0:

        edges.append({
            "source": text1,
            "target": text2,
            "weight": weight,
            "shared_mss": list(shared_mss)
        })
edge_df = pd.DataFrame(edges)
print(edge_df.shape)
display(edge_df)


{'Alexandreis': {'8. B. iv.', '15. A. x.'}, 'Apollonius of Tyre': {'14. C. xi.', '20. C. ii.'}, 'Aspremont': {'15. E. vi.'}, 'Cleriadus et Meliadice': {'20. C. ii.'}, 'Dares Phrygius': {'10. A. x.', '15. A. xxii.', '15. B. xi.', '6. C. viii.', '13. A. v.'}, 'Fierabras': {'15. E. vi.'}, 'Fulk Fitz-Warin': {'12. C. xii.'}, "Guillaume d'Orange": {'20. D. xi.', '20. B. xix.'}, 'Guy of Warwick': {'8. F. ix.'}, 'Havelok': {'13. A. xxi.'}, 'Heroica': {'16. C. xxiii.'}, 'Historia Regum Britannia': {'14. C. i.', '15. C. xvi.'}, 'Historia Trojana': {'12. D. iii.', '16. F. ix.', '13. C. xii.', '15. C. xvi.'}, 'Iliaca': {'16. C. iv. A. B.', '16. D. iii. A. B.'}, 'Karolellus': {'13. A. xviii.'}, "La Vengeance d'Alexandre": {'19. D. i.'}, 'Lancelot du Lac': {'14. E. iii.', '20. B. xix.', '20. D. iv.', '20. C. vi.', '20. D. iii.', '19. B. vii.', '19. C. viii.'}, 'Le Chemin de Vaillance': {'14. E. ii.'}, "Livre de l'Ordre de Chevalerie": {'14. E. ii.'}, 'Mélusine': {'18. B. ii.'}, 'Ogier le Danois': {

,source,target,weight,shared_mss
0,Apollonius of Tyre,Cleriadus et Meliadice,1,[20. C. ii.]
1,Aspremont,Fierabras,1,[15. E. vi.]
2,Aspremont,Ogier le Danois,1,[15. E. vi.]
3,Aspremont,Pontus and Sidoine,1,[15. E. vi.]
4,Aspremont,Quatre fils Aimon,1,[15. E. vi.]
5,Aspremont,Simon de Pouille,1,[15. E. vi.]
6,Dares Phrygius,Prophecy of the Tenth Sibyl,2,"[15. A. xxii., 15. B. xi.]"
7,Fierabras,Ogier le Danois,1,[15. E. vi.]
8,Fierabras,Pontus and Sidoine,1,[15. E. vi.]
9,Fierabras,Quatre fils Aimon,1,[15. E. vi.]


In [7]:
from pyvis.network import Network

#-------------node size---------------
text_freq = df.groupby('title_index')['mss'].nunique()

min_size, max_size = 10, 30

node_sizes = ((text_freq - text_freq.min()) /
             (text_freq.max() - text_freq.min() + 1e-9) *
             (max_size - min_size) + min_size).to_dict()


#-------------node color---------------
from matplotlib import cm, colors

text_theme = df.drop_duplicates('title_index').set_index('title_index')['theme'].to_dict()
text_theme

themes = df['theme'].unique()
cmap = cm.get_cmap('tab10', len(themes))
theme_colors = {
    theme: colors.to_hex(cmap(i))
    for i, theme in enumerate(themes)
}
# {'Classical Romances': '#1f77b4',
#  'British and English Traditions': '#2ca02c',
#  'French Traditions': '#9467bd',
#  'Miscellaneous Romance': '#e377c2',
#  'Allegorical and Didactic Romance': '#bcbd22',
#  'Appendix': '#17becf'}


#----------------pyvis------------------
# notebook=True 在 jupyter 直接显示
net = Network(height='800px', width='100%', notebook=True)

for text in text_freq.index:
    theme = text_theme[text]
    net.add_node(
        text,
        label=text,
        size=node_sizes[text],
        color=theme_colors[theme],
        title=f"{text}, Theme: {theme}"
    )
for _, row in edge_df.iterrows():
    net.add_edge(
        row['source'],
        row['target'],
        value=row['weight']*5,
        color="gray",
        title=f"Shared MSS: {','.join(row['shared_mss'])}"
    )
    
# -------------LAYOUT--------------------
net.force_atlas_2based(
    gravity=-200,          # 整体节点拉开的力度，负值越大，节点分得越开
    central_gravity=0.05,  # 越小，节点越靠中心；越大，节点聚到中心
    spring_length=250,     # 弹簧长度，越大，节点间隔越远
    spring_strength=0.05,  # 弹簧刚度，控制边对节点靠近的作用力
    damping=0.4            # 收敛阻尼
)

#-------------------SHOW---------------------
# 生成 HTML
path_html='../figs/text_in_mss_royal_network.html'
net.show(path_html)

## ----------------add title---------------

# 自动在 body 开始处插入标题
title_text = "Texts Network on Shared Manuscripts colored by Theme"
with open(path_html, 'r', encoding='utf-8') as f:
    html_content = f.read()

# 在 <body> 标签后插入标题
html_content = html_content.replace(
    '<body>',
    f'<body>\n<h2 style="text-align:center">{title_text}</h2>'
)
# 保存回 HTML 文件
with open(path_html, 'w', encoding='utf-8') as f:
    f.write(html_content)

../figs/text_in_mss_royal_network.html


C:\Users\yeliu\AppData\Local\Temp\ipykernel_37132\4227385379.py:20: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('tab10', len(themes))


In [56]:

coll_mss = df.groupby('title_index')['collection'].apply(lambda x: set(x)).to_dict()
print(coll_mss)

from itertools import combinations
edges = []
for text1, text2 in combinations(coll_mss.keys(), 2):

    shared_mss = coll_mss[text1].intersection(coll_mss[text2])

    weight = len(shared_mss)

    if weight > 0:

        edges.append({
            "source": text1,
            "target": text2,
            "weight": weight,
            "shared_collection": list(shared_mss)
        })
edge_df = pd.DataFrame(edges)
print(edge_df.shape)
display(edge_df)


{'Alexander saga': {'Additional'}, 'Alexandreis': {'Additional', 'Royal', 'Harley', 'Burney'}, 'Amadis de Gaula': {'Lansdowne'}, 'Amys and Amylion': {'Harley'}, 'Apollonius of Tyre': {'Royal', 'Sloane', 'Arundel', 'Cotton', 'Additional'}, 'Aronus and Marina': {'Harley'}, 'Artus de Bretagne': {'Additional'}, 'Aspremont': {'Royal', 'Additional'}, 'Battle of Roncevaux': {'Cotton'}, 'Birth of Merlin': {'Arundel'}, "Blanchandin et Orgueillose d'Amors": {'Additional'}, 'Boy and Mantle': {'Additional'}, 'Brutus': {'Cotton'}, 'Cassandra': {'Sloane'}, 'Chanson de Brut': {'Harley'}, 'Cleriadus et Meliadice': {'Royal'}, 'Contes et Fabliaux': {'Additional'}, 'Corbaccio': {'Harley'}, 'Dares Phrygius': {'Royal', 'Sloane', 'Cotton', 'Burney', 'Harley', 'Additional'}, 'Dictys Cretensis': {'Additional', 'Harley', 'Burney'}, 'Doon de la Roche': {'Harley'}, 'Emare': {'Cotton'}, 'Enfances Ogier': {'Harley'}, 'Ereks saga Artuskappa': {'Additional'}, 'Fiammetta': {'Harley'}, 'Fierabras': {'Royal'}, 'Filocol

,source,target,weight,shared_collection
0,Alexander saga,Alexandreis,1,[Additional]
1,Alexander saga,Apollonius of Tyre,1,[Additional]
2,Alexander saga,Artus de Bretagne,1,[Additional]
3,Alexander saga,Aspremont,1,[Additional]
4,Alexander saga,Blanchandin et Orgueillose d'Amors,1,[Additional]
...,...,...,...,...
4157,Uberto and Philomena,Voeux du Paon,1,[Additional]
4158,Uberto and Philomena,Wigalois,1,[Additional]
4159,Vita Merlini,Voeux du Paon,1,[Harley]
4160,Vita Merlini,Ywain and Gawain,1,[Cotton]


In [58]:
from pyvis.network import Network
#-------------node size---------------
text_freq = df.groupby('title_index')['mss'].nunique()

min_size, max_size = 10, 30

node_sizes = ((text_freq - text_freq.min()) /
             (text_freq.max() - text_freq.min() + 1e-9) *
             (max_size - min_size) + min_size).to_dict()


#-------------node color---------------
from matplotlib import cm, colors
text_theme = df.drop_duplicates('title_index').set_index('title_index')['theme'].to_dict()
text_theme

themes = df['theme'].unique()
cmap = cm.get_cmap('tab10', len(themes))
theme_colors = {
    theme: colors.to_hex(cmap(i))
    for i, theme in enumerate(themes)
}
# {'Classical Romances': '#1f77b4',
#  'British and English Traditions': '#2ca02c',
#  'French Traditions': '#9467bd',
#  'Miscellaneous Romance': '#e377c2',
#  'Allegorical and Didactic Romance': '#bcbd22',
#  'Appendix': '#17becf'}


#----------------pyvis------------------
# notebook=True 在 jupyter 直接显示
net = Network(height='800px', width='100%', notebook=True)

for text in text_freq.index:
    theme = text_theme[text]
    net.add_node(
        text,
        label=text,
        size=node_sizes[text],
        color=theme_colors[theme],
        title=f"{text}, Theme: {theme}"
    )
for _, row in edge_df.iterrows():
    net.add_edge(
        row['source'],
        row['target'],
        value=row['weight']*5,
        color="gray",
        title=f"Shared Collection: {','.join(row['shared_collection'])}"
    )
    
# -------------LAYOUT--------------------
net.force_atlas_2based(
    gravity=-200,          # 整体节点拉开的力度，负值越大，节点分得越开
    central_gravity=0.05,  # 越小，节点越靠中心；越大，节点聚到中心
    spring_length=250,     # 弹簧长度，越大，节点间隔越远
    spring_strength=0.05,  # 弹簧刚度，控制边对节点靠近的作用力
    damping=0.4            # 收敛阻尼
)

#-------------------SHOW---------------------
# 生成 HTML
path_html='../figs/text_in_collection_network.html'
net.show(path_html)

## ----------------add title---------------

# 自动在 body 开始处插入标题
title_text = "Texts Network on Shared Collection colored by Theme"
with open(path_html, 'r', encoding='utf-8') as f:
    html_content = f.read()

# 在 <body> 标签后插入标题
html_content = html_content.replace(
    '<body>',
    f'<body>\n<h2 style="text-align:center">{title_text}</h2>'
)
# 保存回 HTML 文件
with open(path_html, 'w', encoding='utf-8') as f:
    f.write(html_content)


C:\Users\yeliu\AppData\Local\Temp\ipykernel_21292\2852827970.py:18: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('tab10', len(themes))


../figs/text_in_collection_network.html


## vis2.2 : connecte les textes s'il sont dans le meme mss, coloré selon le thème


RQ : si  les textes dans le meme manuscript ont le theme?


我们按照mss来连接texts，然后按照theme给texts着色，是不是回答mss是否是按照theme来集合texts的？如果一个cluster中有主导颜色则为是？我现在去掉了theme 为unk的数据，观察到图上大部分的点是没有链接的，但是仍形成了四个简单的小clusters，


In [12]:
df=df_all.copy()
df=df[df["collection"].notna()]
print(len(df_all), len(df))
# print(df.theme.value_counts(dropna=False),"\n")

# nan in "theme":
df=df.dropna(subset='theme')

print(df[["mss",'title_index']].value_counts(dropna=False))

display(df)

583 576
mss               title_index                    
13. A. xxi.       Havelok                            4
Nero A. xi.       Turpin's Chronicle                 4
Harley 4745       Alexandreis                        4
8. B. iv.         Alexandreis                        4
15. A. x.         Alexandreis                        4
                                                    ..
Harley 5662.      Iliaca                             1
13. A. xiv.       Prophecy of the Tenth Sibyl        1
Vespasian E. iv.  Vita Merlini                       1
Vespasian E. x.   Story of Albina and her Sisters    1
Vitellius A. x.   Roman de Brut                      1
Name: count, Length: 331, dtype: int64


,title_theme,title_index,author,theme,cycle,collection,mss,ff,page
0,Heroica,Heroica,Flavius Philostratus,Classical Romances,Cycle of Troy,Royal,16. C. xxiii.,ff. 2-69 b,1
1,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Royal,16. C. iv. A. B.,NaN,2
2,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Royal,16. D. iii. A. B.,NaN,2
3,Iliaca,Iliaca,Joannes Tzetzes,Classical Romances,Cycle of Troy,Harley,Harley 5662.,ff. 1-56,2
4,Dictys Cretensis,Dictys Cretensis,NaN,Classical Romances,Cycle of Troy,Harley,Harley 3514,NaN,9
...,...,...,...,...,...,...,...,...,...
404,Turpin's Chronicle,Turpin's Chronicle,NaN,Appendix,NaN,Additional,Add. 6924.,ff. 297-303,950
405,Turpin's Chronicle,Turpin's Chronicle,NaN,Appendix,NaN,Additional,"Add. 12,213.",ff. 1-184 b,950
406,Turpin's Chronicle,Turpin's Chronicle,NaN,Appendix,NaN,Additional,"Add. 17,920.",ff. 6 b-19 b,950
407,Turpin's Chronicle,Turpin's Chronicle,NaN,Appendix,NaN,Additional,"Add. 19,513.",ff. 141-163,950


In [13]:
text_mss = df.groupby('title_index')['mss'].apply(lambda x: set(x)).to_dict()
print(text_mss)

from itertools import combinations
edges = []
for text1, text2 in combinations(text_mss.keys(), 2):

    shared_mss = text_mss[text1].intersection(text_mss[text2])

    weight = len(shared_mss)

    if weight > 0:

        edges.append({
            "source": text1,
            "target": text2,
            "weight": weight,
            "shared_mss": list(shared_mss)
        })
edge_df = pd.DataFrame(edges)
print(edge_df.shape)
display(edge_df)


{'Alexander saga': {'Add. 24,969.', 'Add. 11,238.'}, 'Alexandreis': {'Add. 18,217', '8. B. iv.', 'Add. 17,084', 'Add. 22,821', 'Burney 312', 'Harley 5437', '15. A. x.', 'Harley 4745'}, 'Amadis de Gaula': {'Lansdowne 766.'}, 'Amys and Amylion': {'Harley 2386.'}, 'Apollonius of Tyre': {'Vespasian A. xiii.', 'Sloane 2233', 'Arundel 292.', '20. C. ii.', 'Titus D. iii.', 'Arundel 123.', 'Sloane 1619', '14. C. xi.', 'Add. 4864.', 'Add. 4857.'}, 'Aronus and Marina': {'Harley 2678.'}, 'Artus de Bretagne': {'Add. 10,295'}, 'Aspremont': {'Add. 10,508.', '15. E. vi.'}, 'Battle of Roncevaux': {'Titus A. xix.'}, 'Birth of Merlin': {'Arundel 220. Art. L'}, "Blanchandin et Orgueillose d'Amors": {'Add. 15,212.'}, 'Boy and Mantle': {'Add. 27879'}, 'Brutus': {'Vespasian A. x.'}, 'Cassandra': {'Sloane 457'}, 'Chanson de Brut': {'Harley 1605. Art. I.'}, 'Cleriadus et Meliadice': {'20. C. ii.'}, 'Contes et Fabliaux': {'Add. 15,210-15,213'}, 'Corbaccio': {'Harley 3581'}, 'Dares Phrygius': {'Harley 641.', 'V

,source,target,weight,shared_mss
0,Amys and Amylion,Story of Albina and her Sisters,1,[Harley 2386.]
1,Apollonius of Tyre,Cleriadus et Meliadice,1,[20. C. ii.]
2,Apollonius of Tyre,Dares Phrygius,1,[Sloane 1619]
3,Apollonius of Tyre,Prophecy of the Tenth Sibyl,1,[Titus D. iii.]
4,Apollonius of Tyre,Turpin's Chronicle,1,[Vespasian A. xiii.]
...,...,...,...,...
76,Story of Albina and her Sisters,Turpin's Chronicle,1,[Titus A. xix.]
77,Story of Albina and her Sisters,Vita Merlini,1,[Titus A. xix.]
78,"Tinctoris, Compostella",Turpin's Chronicle,1,"[Add. 12,213.]"
79,Trójumanna saga,Two Romantic Tales,1,[Add. 4869.]


In [14]:
text_coll = df.drop_duplicates('title_index').set_index('title_index')['collection'].to_dict()
text_coll

{'Heroica': 'Royal',
 'Iliaca': 'Royal',
 'Dictys Cretensis': 'Harley',
 'Dares Phrygius': 'Royal',
 'Roman de Troie': 'Harley',
 'Historia Trojana': 'Royal',
 'Trójumanna saga': 'Additional',
 'Rimur from Trójumanna saga': 'Additional',
 'History of Troy': 'Burney',
 'Filostrato': 'Additional',
 'Troilus and Cryseyde': 'Harley',
 'Troy Book': 'Royal',
 "Roman d'Eneas": 'Additional',
 'Romance of Troy': 'Harley',
 'Romance of Thebes': 'Royal',
 'Romance of Jason': 'Additional',
 'Alexandreis': 'Royal',
 'Alexander saga': 'Additional',
 'Marvels of India': 'Cotton',
 'Treatise on the Brahmins': 'Arundel',
 'Letters of Dindimus to Alexander the Great': 'Cotton',
 "La Vengeance d'Alexandre": 'Royal',
 'Voeux du Paon': 'Harley',
 'Florimont': 'Harley',
 'Apollonius of Tyre': 'Royal',
 'Sir Orpheo': 'Harley',
 "Roman d'Athis et Profilas": 'Additional',
 'Titus and Vespasian': 'Royal',
 'Prophecy of the Tenth Sibyl': 'Royal',
 'Life of Virgilius': 'Additional',
 'Story of Albina and her Sist

In [28]:
from pyvis.network import Network

#-------------node size---------------
text_freq = df.groupby('title_index')['mss'].nunique()

min_size, max_size = 10, 30

node_sizes = ((text_freq - text_freq.min()) /
             (text_freq.max() - text_freq.min() + 1e-9) *
             (max_size - min_size) + min_size).to_dict()


#-------------node color---------------
from matplotlib import cm, colors

text_theme = df.drop_duplicates('title_index').set_index('title_index')['theme'].to_dict()
text_theme

themes = df['theme'].unique()
cmap = cm.get_cmap('tab20', len(themes))
theme_colors = {
    theme: colors.to_hex(cmap(i))
    for i, theme in enumerate(themes)
}


#-------------text color----------------
text_coll = df.drop_duplicates('title_index').set_index('title_index')['collection'].to_dict()

colls = df['collection'].unique()
cmap = cm.get_cmap('Dark2', len(themes))
coll_colors = {
    colls: colors.to_hex(cmap(i))
    for i, colls in enumerate(colls)
}

#----------------pyvis------------------
# notebook=True 在 jupyter 直接显示
net = Network(height='800px', width='100%', notebook=True)

for text in text_freq.index:
    theme = text_theme[text]
    coll= text_coll[text]
    net.add_node(
        text,
        label=text,
        font={"color":coll_colors[coll]},
        size=node_sizes[text],
        color=theme_colors[theme],
        title=f"{text}, Theme: {theme}",
        # group=coll,   # 👈关键

    )
for _, row in edge_df.iterrows():
    net.add_edge(
        row['source'],
        row['target'],
        value=row['weight']*5,
        color="gray",
        title=f"Shared MSS: {','.join(row['shared_mss'])}"
    )
    
# -------------LAYOUT--------------------
net.force_atlas_2based(
    gravity=-200,          # 整体节点拉开的力度，负值越大，节点分得越开
    central_gravity=0.05,  # 越小，节点越靠中心；越大，节点聚到中心
    spring_length=250,     # 弹簧长度，越大，节点间隔越远
    spring_strength=0.05,  # 弹簧刚度，控制边对节点靠近的作用力
    damping=0.4            # 收敛阻尼
)        
    
#-------------------SHOW---------------------
# 生成 HTML
path_html='../figs/text_in_mss_network.html'
net.show(path_html)

## ----------------add title---------------

# 自动在 body 开始处插入标题
title_text = "Texts Network on Shared Manuscripts colored by Theme"
with open(path_html, 'r', encoding='utf-8') as f:
    html_content = f.read()

# 在 <body> 标签后插入标题
html_content = html_content.replace(
    '<body>',
    f'<body>\n<h2 style="text-align:center">{title_text}</h2>'
)
# 保存回 HTML 文件
with open(path_html, 'w', encoding='utf-8') as f:
    f.write(html_content)


../figs/text_in_mss_network.html


C:\Users\yeliu\AppData\Local\Temp\ipykernel_37132\3397188283.py:20: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('tab20', len(themes))
C:\Users\yeliu\AppData\Local\Temp\ipykernel_37132\3397188283.py:31: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('Dark2', len(themes))
